# Exercise - Multi step Workflow - STARTER

In this exercise, you’ll build a multi-step workflow using LCEL to solve a more complex task than simply generating a joke. 

**Challenge**

Create an AI Business Advisor that:

1. Accepts an industry as input.
2. Generates a business idea.
3. Analyzes the strengths and weaknesses.
4. Formats the results as a final report.


## 0. Import the necessary libs

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field

## 1. Instantiate Chat Model with your API Key

To be able to connect with OpenAI, you need to instantiate an ChatOpenAI client passing your OpenAI key.

You can pass the `api_key` argument directly.
```python
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key="voc-",
)
```

In [2]:
# TODO - Instantiate your chat model
import os
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key =os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

## 2. Your first Chain

In the end of each chain, you should parse the output and save the logs

In [3]:
logs = []

In [4]:
parser = StrOutputParser()

In [5]:
parse_and_log_output_chain = RunnableParallel(
    output=parser, 
    log=RunnableLambda(lambda x: logs.append(x))
)

## 3. Idea Generation

Craft a prompt to generate a business idea for the given industry. 

Make sure {industry} placeholder is inside your template, so it can be filled when the chain is invoked.

In [6]:
# TODO - Your prompt Template
idea_prompt = PromptTemplate(
    template=(
        "Generate a business idea for {industry}"
    )
)

In [7]:
# TODO - Create your idea_chain: idea_prompt -> llm -> parse_and_log_output_chain
idea_chain = idea_prompt | llm | parse_and_log_output_chain

In [8]:
# TODO - Test your idea_chain invoking it by passing an industy like "agro" to it
idea_result = idea_chain.invoke("agro")

In [9]:
idea_result["output"]

"**Business Idea: Smart Vertical Farming Solutions**\n\n**Overview:**\nThe Smart Vertical Farming Solutions business focuses on creating automated, modular vertical farming systems that utilize advanced technology to optimize space, resources, and crop yield. This business targets urban areas where space is limited and the demand for fresh, locally-sourced produce is high.\n\n**Key Components:**\n\n1. **Modular Vertical Farming Units:**\n   - Design and manufacture compact, stackable vertical farming units that can be easily installed in urban settings, such as rooftops, warehouses, or even small backyards.\n   - Each unit would be equipped with hydroponic or aeroponic systems to grow a variety of crops, including leafy greens, herbs, and small fruits.\n\n2. **Smart Technology Integration:**\n   - Implement IoT (Internet of Things) sensors to monitor environmental conditions (light, temperature, humidity, nutrient levels) and automate irrigation and nutrient delivery.\n   - Develop a u

In [10]:
logs

[AIMessage(content="**Business Idea: Smart Vertical Farming Solutions**\n\n**Overview:**\nThe Smart Vertical Farming Solutions business focuses on creating automated, modular vertical farming systems that utilize advanced technology to optimize space, resources, and crop yield. This business targets urban areas where space is limited and the demand for fresh, locally-sourced produce is high.\n\n**Key Components:**\n\n1. **Modular Vertical Farming Units:**\n   - Design and manufacture compact, stackable vertical farming units that can be easily installed in urban settings, such as rooftops, warehouses, or even small backyards.\n   - Each unit would be equipped with hydroponic or aeroponic systems to grow a variety of crops, including leafy greens, herbs, and small fruits.\n\n2. **Smart Technology Integration:**\n   - Implement IoT (Internet of Things) sensors to monitor environmental conditions (light, temperature, humidity, nutrient levels) and automate irrigation and nutrient delivery

## 4. Idea Analysis

Craft a prompt to analyze the generated idea's strengths and weaknesses

In [11]:
# TODO - Your prompt Template
analysis_prompt = PromptTemplate(
    template=(
        "Analyze the generated idea's strenghts and weaknesses:\n{idea}"
    )
)

In [12]:
# TODO - Your chain
analysis_chain = (
    analysis_prompt | llm | parse_and_log_output_chain
)

In [13]:
# TODO - Test your analysis_chain invoking it by passing idea_result["output"] to it
analysis_result = analysis_chain.invoke(idea_result["output"])

In [14]:
analysis_result["output"]

'### Strengths\n\n1. **Innovative Concept:**\n   - The idea of modular vertical farming units is innovative and addresses the growing need for sustainable food production in urban areas where space is limited.\n\n2. **Technology Integration:**\n   - Utilizing IoT technology for monitoring and automation enhances efficiency and allows users to manage their farms remotely, appealing to tech-savvy consumers.\n\n3. **Sustainability Focus:**\n   - The emphasis on renewable energy and water recycling aligns with current trends towards sustainability and environmental responsibility, attracting eco-conscious customers.\n\n4. **Community Engagement:**\n   - Offering workshops and educational programs fosters community involvement and builds a loyal customer base, while also promoting awareness of sustainable practices.\n\n5. **Diverse Revenue Streams:**\n   - Multiple revenue streams, including sales of farming units, subscriptions, workshops, and partnerships, can provide financial stability 

In [15]:
logs

[AIMessage(content="**Business Idea: Smart Vertical Farming Solutions**\n\n**Overview:**\nThe Smart Vertical Farming Solutions business focuses on creating automated, modular vertical farming systems that utilize advanced technology to optimize space, resources, and crop yield. This business targets urban areas where space is limited and the demand for fresh, locally-sourced produce is high.\n\n**Key Components:**\n\n1. **Modular Vertical Farming Units:**\n   - Design and manufacture compact, stackable vertical farming units that can be easily installed in urban settings, such as rooftops, warehouses, or even small backyards.\n   - Each unit would be equipped with hydroponic or aeroponic systems to grow a variety of crops, including leafy greens, herbs, and small fruits.\n\n2. **Smart Technology Integration:**\n   - Implement IoT (Internet of Things) sensors to monitor environmental conditions (light, temperature, humidity, nutrient levels) and automate irrigation and nutrient delivery

## 5. Report Generation

Craft a prompt to structure the information into a business report.

In [16]:
# TODO - Your prompt Template
report_prompt = PromptTemplate.from_template(
"You are given a business idea analysis.\n"
"Analysis:\n{output}\n\n"
"Extract the strengths and weaknesses from this analysis."
)

In [17]:
class AnalysisReport(BaseModel):
    """Strengths and Weaknesses about a business idea"""
    strengths: list[str] = Field(default_factory=list, description="Idea's strength list")
    weaknesses: list[str] = Field(default_factory=list, description="Idea's weakness list")

In [18]:
report_chain = (
    report_prompt | llm.with_structured_output(schema=AnalysisReport, method="function_calling")
)

In [19]:
# TODO - Test your report_chain invoking it by passing analysis_result["output"] to it
report_result = report_chain.invoke({"output": analysis_result["output"]})

In [20]:
report_result

AnalysisReport(strengths=['Innovative Concept: The idea of modular vertical farming units is innovative and addresses the growing need for sustainable food production in urban areas where space is limited.', 'Technology Integration: Utilizing IoT technology for monitoring and automation enhances efficiency and allows users to manage their farms remotely, appealing to tech-savvy consumers.', 'Sustainability Focus: The emphasis on renewable energy and water recycling aligns with current trends towards sustainability and environmental responsibility, attracting eco-conscious customers.', 'Community Engagement: Offering workshops and educational programs fosters community involvement and builds a loyal customer base, while also promoting awareness of sustainable practices.', 'Diverse Revenue Streams: Multiple revenue streams, including sales of farming units, subscriptions, workshops, and partnerships, can provide financial stability and growth opportunities.', 'Market Demand: The increasi

## 6. End to End Chain

In [21]:
e2e_chain = ( 
    RunnablePassthrough() 
    | idea_chain
    | RunnableParallel(idea=RunnablePassthrough())
    | analysis_chain
    | report_chain
)

In [22]:
e2e_chain.get_graph().print_ascii()

            +------------------+         
            | PassthroughInput |         
            +------------------+         
                      *                  
                      *                  
                      *                  
              +-------------+            
              | Passthrough |            
              +-------------+            
                      *                  
                      *                  
                      *                  
             +----------------+          
             | PromptTemplate |          
             +----------------+          
                      *                  
                      *                  
                      *                  
               +------------+            
               | ChatOpenAI |            
               +------------+            
                      *                  
                      *                  
                      *           

In [23]:
# Change the industry if you want
e2e_result = e2e_chain.invoke("agro")

In [24]:
e2e_result

AnalysisReport(strengths=['Innovative Concept: The idea of smart vertical farming is at the intersection of technology and agriculture, addressing modern challenges such as urbanization and food security.', 'Sustainability Focus: The emphasis on sustainable practices, such as renewable energy and water recycling, aligns with growing consumer demand for environmentally friendly products and practices.', 'Market Demand: There is a rising trend for locally-sourced, fresh produce, especially in urban areas. This business model caters to health-conscious consumers and those interested in sustainability.', 'Modular Design: The modular vertical farming units can be easily adapted to various urban environments, making them versatile and scalable.', 'Smart Technology Integration: Utilizing IoT and mobile applications enhances user experience and allows for real-time monitoring and optimization, which can lead to better crop yields.', 'Community Engagement: Offering workshops and educational pro

In [25]:
e2e_result.strengths

['Innovative Concept: The idea of smart vertical farming is at the intersection of technology and agriculture, addressing modern challenges such as urbanization and food security.',
 'Sustainability Focus: The emphasis on sustainable practices, such as renewable energy and water recycling, aligns with growing consumer demand for environmentally friendly products and practices.',
 'Market Demand: There is a rising trend for locally-sourced, fresh produce, especially in urban areas. This business model caters to health-conscious consumers and those interested in sustainability.',
 'Modular Design: The modular vertical farming units can be easily adapted to various urban environments, making them versatile and scalable.',
 'Smart Technology Integration: Utilizing IoT and mobile applications enhances user experience and allows for real-time monitoring and optimization, which can lead to better crop yields.',
 'Community Engagement: Offering workshops and educational programs can foster com

In [26]:
e2e_result.weaknesses

['High Initial Investment: The cost of developing and manufacturing advanced vertical farming systems, along with the technology integration, may require significant upfront capital.',
 'Technical Challenges: Implementing and maintaining smart technology and IoT systems can be complex and may require specialized knowledge, which could lead to operational challenges.',
 'Market Competition: The vertical farming industry is becoming increasingly competitive, with established players and new entrants. Differentiating the business may be challenging.',
 'Consumer Education: While there is a growing interest in sustainable practices, some consumers may still be unaware of the benefits of vertical farming, necessitating additional marketing and education efforts.',
 'Regulatory Hurdles: Urban farming initiatives may face zoning laws, health regulations, and other bureaucratic challenges that could hinder operations.',
 'Dependency on Technology: The reliance on technology for monitoring and 

In [27]:
logs

[AIMessage(content="**Business Idea: Smart Vertical Farming Solutions**\n\n**Overview:**\nThe Smart Vertical Farming Solutions business focuses on creating automated, modular vertical farming systems that utilize advanced technology to optimize space, resources, and crop yield. This business targets urban areas where space is limited and the demand for fresh, locally-sourced produce is high.\n\n**Key Components:**\n\n1. **Modular Vertical Farming Units:**\n   - Design and manufacture compact, stackable vertical farming units that can be easily installed in urban settings, such as rooftops, warehouses, or even small backyards.\n   - Each unit would be equipped with hydroponic or aeroponic systems to grow a variety of crops, including leafy greens, herbs, and small fruits.\n\n2. **Smart Technology Integration:**\n   - Implement IoT (Internet of Things) sensors to monitor environmental conditions (light, temperature, humidity, nutrient levels) and automate irrigation and nutrient delivery

## 7. Experiment

Now that you understood how it works, experiment with new things.
- Improve memory
- Explore the Runnables
- Add More Complexity